In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'Thessaly'

In [4]:
YEAR = 2023
MONTH = 'June'
PERIOD = '2nd'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.76105,39.69186,2023-06-16,θεσσαλιας,αγιας,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,661.8,10705.0,17.3,45.75488,0.576611,0.243522,-0.480992,-0.243522,0.540359,0.232554,-0.455349,-0.232554,0.041794,0.033826,0.034503,0.033826,19.820383,24.286250,15.354516,10.418186,4.056923,10.285211,3.423839,16.670680,6.227661,17.857476,7.916795,47.538768,52.504310,179.677592,8282.127022,1331.619869,1,123.585589,121.767859,183.036108,0.0,21.311709,31,90,30,90.0,30,90,10,10,1,6,6,2,0,22,0,0
1,22.72671,39.12560,2023-06-16,θεσσαλιας,αλμυρου,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,905.4,16004.0,20.6,45.24728,0.432729,0.195651,-0.365570,-0.195651,0.408731,0.191460,-0.345646,-0.191460,0.036262,0.034202,0.026982,0.034202,21.116458,27.468333,14.764583,11.219310,3.845804,10.731281,3.006374,16.701534,6.377584,19.115966,8.094541,41.958648,43.037496,276.036993,10017.207897,1610.666116,14,259.585198,342.571837,171.163634,0.0,3.415750,11,80,10,72.0,10,72,1,1,7,1,1,2,0,16,0,0
2,23.99842,39.24585,2023-06-16,θεσσαλιας,αλοννησου,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,129.6,3153.0,21.2,46.00175,-0.176781,0.153420,0.340381,-0.153420,-0.193634,0.198027,0.371268,-0.198027,0.024820,0.064921,0.042074,0.064921,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.300147,1.639872,255.294237,1484.683259,55302.082778,24,302.637680,165.397106,195.027810,0.0,1.676994,21,94,20,94.0,20,94,8,8,4,1,1,2,0,1,0,0
3,21.48510,39.29630,2023-06-16,θεσσαλιας,αργιθεας,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,372.9,3515.0,9.3,44.78626,0.514500,0.228331,-0.410671,-0.228331,0.467199,0.231124,-0.367177,-0.231124,0.080654,0.056288,0.060995,0.056288,16.303077,21.382308,11.223846,6.942786,1.831897,7.062325,-0.605892,13.120876,3.695211,13.925591,4.285066,19.744202,23.864535,217.606597,19327.278832,1925.318354,27,284.268894,1127.254453,234.678996,0.0,1.719393,21,88,20,88.0,20,88,8,8,4,1,1,2,0,19,0,0
4,22.93502,39.38117,2023-06-16,θεσσαλιας,βολου,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,385.6,138865.0,374.6,45.57293,0.391943,0.158412,-0.329575,-0.158412,0.374578,0.139134,-0.325931,-0.139134,0.034830,0.026475,0.027319,0.026475,22.264091,30.020000,14.508182,12.159420,4.198851,11.394699,3.845437,17.241419,5.994943,19.899661,8.124530,16.179357,19.405314,198.751209,5935.142744,3071.988689,5,143.181190,234.931813,174.426021,0.0,4.643743,31,91,30,91.0,30,91,10,10,1,6,6,2,0,26,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

scaler = MinMaxScaler()
imputer = KNNImputer()

X_test = scaler.fit_transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.34107,39.61048,λαρισαιων,16,6,2023,0.989606
1,22.29611,39.76297,τυρναβου,16,6,2023,0.954657
2,22.09265,39.48079,παλαμα,16,6,2023,0.918827
3,22.02629,39.60300,φαρκαδονας,16,6,2023,0.715102
4,22.54288,39.85090,τεμπων,16,6,2023,0.562422
5,21.89535,39.26951,καρδιτσας,16,6,2023,0.314416
6,22.08995,39.24048,σοφαδων,16,6,2023,0.196449
7,22.93502,39.38117,βολου,16,6,2023,0.185295
8,22.15952,39.95174,ελασσονας,16,6,2023,0.167917
9,22.80830,39.43236,ρηγα φερραιου,16,6,2023,0.113865


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0079983711252831, 0.0857458244919643, 0.5105258673268654, 0.8201202051121793, 0.9298314119184128, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.34107,39.61048,λαρισαιων,16,6,2023,0.989606,5
1,22.29611,39.76297,τυρναβου,16,6,2023,0.954657,5
2,22.09265,39.48079,παλαμα,16,6,2023,0.918827,4
3,22.02629,39.60300,φαρκαδονας,16,6,2023,0.715102,3
4,22.54288,39.85090,τεμπων,16,6,2023,0.562422,3
5,21.89535,39.26951,καρδιτσας,16,6,2023,0.314416,2
6,22.08995,39.24048,σοφαδων,16,6,2023,0.196449,2
7,22.93502,39.38117,βολου,16,6,2023,0.185295,2
8,22.15952,39.95174,ελασσονας,16,6,2023,0.167917,2
9,22.80830,39.43236,ρηγα φερραιου,16,6,2023,0.113865,2


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results